In [ ]:
import pandas as pd
import sqlite3
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from collections import Counter
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Ensure necessary NLTK data is downloaded
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# ==========================================
# 1. MOCK "SCRAPED" DATASET (For Demonstration)
# ==========================================
data = {
    'id': [1, 2, 3],
    'raw_html': [
        "<html><body><h1>Python Data!</h1> <p>Data science is amazing. I love analyzing data!</p></body></html>",
        "<div><b>Machine Learning:</b> It's the future of artificial intelligence... 100% real.</div>",
        "<p>Natural Language Processing (NLP) helps computers understand human language better.</p>"
    ]
}
df = pd.DataFrame(data)

# ==========================================
# 2. TEXT CLEANING PIPELINE
# ==========================================
def clean_text_pipeline(text):
    """Cleans HTML, punctuation, lowercases, tokenizes, and removes stopwords."""
    # 1. Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 2. Remove URLs (if any exist)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # 3. Remove punctuation, numbers, and unwanted characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 4. Convert to lowercase
    text = text.lower()

    # 5. Tokenize text
    tokens = word_tokenize(text)

    # 6. Remove stopwords
    stop_words = set(stopwords.words('english'))
    cleaned_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]

    return cleaned_tokens

# Apply the pipeline to create new columns for tokens and joined text
df['cleaned_tokens'] = df['raw_html'].apply(clean_text_pipeline)
df['cleaned_text'] = df['cleaned_tokens'].apply(lambda x: ' '.join(x))

# ==========================================
# 3. CALCULATE FREQUENCIES & BIGRAMS
# ==========================================
# Flatten the list of tokens to get global frequencies
all_words = [word for tokens in df['cleaned_tokens'] for word in tokens]
word_freq = Counter(all_words)

# Calculate Bigrams
all_bigrams = []
for tokens in df['cleaned_tokens']:
    all_bigrams.extend(list(ngrams(tokens, 2)))
bigram_freq = Counter(all_bigrams)

# Convert Word Frequencies to a DataFrame (Keyword Table)
keyword_df = pd.DataFrame(word_freq.items(), columns=['Keyword', 'Frequency']).sort_values(by='Frequency', ascending=False)

# Convert Bigram Frequencies to a DataFrame
bigram_df = pd.DataFrame([(' '.join(bg), freq) for bg, freq in bigram_freq.items()],
                         columns=['Bigram', 'Frequency']).sort_values(by='Frequency', ascending=False)

# ==========================================
# 4. VISUALIZATION: WORD CLOUD & PLOTS
# ==========================================
def plot_visualizations(word_freq, keyword_df):
    plt.figure(figsize=(15, 6))

    # Plot 1: Word Cloud
    plt.subplot(1, 2, 1)
    wordcloud = WordCloud(width=800, height=400, background_color='white', colormap='viridis').generate_from_frequencies(word_freq)
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Word Cloud of Cleaned Text', fontsize=16)

    # Plot 2: Top 10 Word Frequencies Bar Plot
    plt.subplot(1, 2, 2)
    top_10 = keyword_df.head(10)
    sns.barplot(x='Frequency', y='Keyword', data=top_10, palette='magma')
    plt.title('Top 10 Most Frequent Words', fontsize=16)
    plt.xlabel('Frequency')
    plt.ylabel('Keyword')

    plt.tight_layout()
    plt.show()

plot_visualizations(word_freq, keyword_df)

# ==========================================
# 5. SAVE DATASETS (CSV & SQLite)
# ==========================================
# Save to CSV
df.drop(columns=['cleaned_tokens']).to_csv('cleaned_dataset.csv', index=False)
keyword_df.to_csv('keyword_frequencies.csv', index=False)
print("Data saved to CSV files: 'cleaned_dataset.csv' & 'keyword_frequencies.csv'.")

# Save to SQLite Database
db_name = 'scraped_data.db'
conn = sqlite3.connect(db_name)

# Save main cleaned dataset to SQLite (lists are not supported in SQLite, so we drop the raw token list)
df_to_sql = df.drop(columns=['cleaned_tokens'])
df_to_sql.to_sql('cleaned_articles', conn, if_exists='replace', index=False)

# Save Keyword table to SQLite
keyword_df.to_sql('keyword_table', conn, if_exists='replace', index=False)

# Save Bigram table to SQLite
bigram_df.to_sql('bigram_table', conn, if_exists='replace', index=False)

conn.close()
print(f"Data and Keyword Tables successfully saved to SQLite database: '{db_name}'.")

In [ ]:

# ============================================================
# MEMBER 3: TF-IDF KEYWORD EXTRACTION AND TOPIC MODELLING
# Project: Keyword Identification
#
# Purpose:
# 1. Extract important keywords using TF-IDF
# 2. Compare keywords across two sources
# 3. Discover themes using NMF topic modelling
# 4. Export results for Member 4
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

# Set a random seed so that the NMF results are reproducible.
RANDOM_STATE = 42

# ------------------------------------------------------------
# SETTINGS YOU MAY CHANGE LATER
# ------------------------------------------------------------

# Number of top keywords to display for each article.
TOP_KEYWORDS_PER_ARTICLE = 10

# Number of keywords to display for each source.
TOP_KEYWORDS_PER_SOURCE = 30

# Number of words to display for each discovered topic.
TOP_WORDS_PER_TOPIC = 15

# Initial number of topics for NMF.
# We can test different values later.
N_TOPICS = 5

# Folder where all your output files will be saved.
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully.")
print("Output folder is ready.")


Libraries imported successfully.
Output folder is ready.


In [ ]:

# ============================================================
# LOAD THE CLEANED DATASET
# ============================================================

from google.colab import files

# This opens a file-upload window in Google Colab.
# Select the cleaned CSV file provided by Member 2.
uploaded = files.upload()

# Get the name of the uploaded file automatically.
filename = next(iter(uploaded))

# Read the CSV into a Pandas DataFrame.
df = pd.read_csv(filename)

# Display basic information about the dataset.
print("Dataset loaded:", filename)
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

# Show the first five articles.
display(df.head())


In [ ]:

# ============================================================
# CHECK THE REQUIRED COLUMNS AND PREPARE THE DATA
# ============================================================

# ------------------------------------------------------------
# EDIT THESE TWO LINES ONLY IF MEMBER 2 USED DIFFERENT NAMES.
# For example, if the cleaned text column is called "text",
# change TEXT_COLUMN = "clean_text" to TEXT_COLUMN = "text".
# ------------------------------------------------------------

TEXT_COLUMN = "clean_text"
SOURCE_COLUMN = "source"

# The title column is optional.
TITLE_COLUMN = "title"

# Check that the required columns exist.
required_columns = [TEXT_COLUMN, SOURCE_COLUMN]

for column in required_columns:
    if column not in df.columns:
        raise ValueError(
            f"Required column '{column}' not found. "
            f"Available columns: {df.columns.tolist()}. "
            "Edit TEXT_COLUMN or SOURCE_COLUMN above."
        )

# Remove rows with missing source labels or text.
df = df.dropna(subset=required_columns).copy()

# Convert the text and source labels into strings.
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str)
df[SOURCE_COLUMN] = df[SOURCE_COLUMN].astype(str)

# Remove rows where the cleaned text is empty.
df = df[df[TEXT_COLUMN].str.strip() != ""].copy()

# Reset the row index so that it runs from 0 to n-1.
df = df.reset_index(drop=True)

# Create standard column names for the rest of our code.
# This avoids repeatedly having to remember Member 2's
# original column names.
df["clean_text"] = df[TEXT_COLUMN]
df["source"] = df[SOURCE_COLUMN]

# If there is no title column, create a placeholder.
if TITLE_COLUMN in df.columns:
    df["title"] = df[TITLE_COLUMN].fillna("").astype(str)
else:
    df["title"] = ""

print("Dataset preparation complete.")
print("Number of usable articles:", len(df))

print("\nArticles per source:")
print(df["source"].value_counts())

print("\nMissing values in key columns:")
print(df[["source", "clean_text", "title"]].isna().sum())

# We expect two sources for this project.
print("\nNumber of unique sources:", df["source"].nunique())

if df["source"].nunique() != 2:
    print(
        "WARNING: Check the source labels. "
        "This project is intended to compare two sources."
    )


In [ ]:

# ============================================================
# TF-IDF KEYWORD EXTRACTION
# ============================================================

# TF-IDF stands for Term Frequency–Inverse Document Frequency.
#
# TF measures how prominent a word is within a document.
# IDF reduces the weight of words that occur in many documents.
#
# The result is a numerical matrix:
# Rows    = articles
# Columns = words or phrases
# Values  = TF-IDF scores

vectorizer = TfidfVectorizer(
    max_df=0.95,          # Ignore terms in more than 95% of documents
    min_df=2,              # Keep terms appearing in at least 2 documents
    ngram_range=(1, 2),    # Include single words and two-word phrases
    max_features=5000,     # Limit vocabulary size
    sublinear_tf=True      # Reduce the effect of repeated word counts
)

# Learn the vocabulary and calculate TF-IDF scores.
X_tfidf = vectorizer.fit_transform(df["clean_text"])

# Retrieve the words/phrases corresponding to matrix columns.
terms = vectorizer.get_feature_names_out()

print("TF-IDF calculation complete.")
print("Number of articles:", X_tfidf.shape[0])
print("Number of terms:", X_tfidf.shape[1])
print("Matrix dimensions:", X_tfidf.shape)


In [ ]:

# ============================================================
# EXTRACT TOP KEYWORDS FOR EACH ARTICLE
# ============================================================

def extract_top_keywords(matrix, vocabulary, row_index, top_n=10):
    """
    Extract the highest-scoring TF-IDF terms for one article.

    Parameters
    ----------
    matrix:
        TF-IDF matrix containing all articles.
    vocabulary:
        Array of terms corresponding to the matrix columns.
    row_index:
        Index of the article we want to analyse.
    top_n:
        Maximum number of keywords to return.

    Returns
    -------
    List of (keyword, score) pairs, sorted from highest score
    to lowest score.
    """

    # Get the TF-IDF scores for the selected article.
    scores = matrix[row_index].toarray().ravel()

    # Sort term indices by descending score.
    sorted_indices = np.argsort(scores)[::-1]

    # Keep only terms with a positive score.
    selected_indices = [
        i for i in sorted_indices if scores[i] > 0
    ][:top_n]

    # Return the keyword and its score.
    return [
        (vocabulary[i], round(float(scores[i]), 4))
        for i in selected_indices
    ]


# Create a list to store all article-keyword results.
article_keyword_rows = []

# Process every article in the dataset.
for row_index in range(X_tfidf.shape[0]):

    top_keywords = extract_top_keywords(
        X_tfidf,
        terms,
        row_index,
        top_n=TOP_KEYWORDS_PER_ARTICLE
    )

    # Save each keyword as a separate row.
    for keyword, score in top_keywords:
        article_keyword_rows.append({
            "article_id": row_index,
            "source": df.loc[row_index, "source"],
            "title": df.loc[row_index, "title"],
            "keyword": keyword,
            "tfidf_score": score
        })

# Convert the results into a DataFrame.
article_keywords_df = pd.DataFrame(article_keyword_rows)

print("Top article keywords extracted.")
display(article_keywords_df.head(20))


In [ ]:

# ============================================================
# SOURCE-LEVEL KEYWORD COMPARISON
# ============================================================

# We calculate the average TF-IDF score of each term across
# all articles belonging to each source.
#
# This provides a useful source-level summary.
# However, a high average score does not automatically mean
# a word is unique to that source.

source_keyword_rows = []

for source_name in df["source"].unique():

    # Select articles belonging to the current source.
    source_mask = (
        df["source"] == source_name
    ).to_numpy()

    source_matrix = X_tfidf[source_mask]

    # Calculate the mean score of each term across these articles.
    mean_scores = np.asarray(
        source_matrix.mean(axis=0)
    ).ravel()

    # Sort terms from highest mean score to lowest.
    sorted_indices = np.argsort(mean_scores)[::-1]

    # Keep the requested number of top terms.
    top_indices = [
        i for i in sorted_indices if mean_scores[i] > 0
    ][:TOP_KEYWORDS_PER_SOURCE]

    for i in top_indices:
        source_keyword_rows.append({
            "source": source_name,
            "keyword": terms[i],
            "mean_tfidf": round(float(mean_scores[i]), 5)
        })

source_keywords_df = pd.DataFrame(source_keyword_rows)

print("Source-level keyword comparison complete.")
display(source_keywords_df.head(20))


In [ ]:

# ============================================================
# PLOT THE TOP KEYWORDS FOR EACH SOURCE
# ============================================================

for source_name in df["source"].unique():

    # Select the top 15 terms for this source.
    plot_data = (
        source_keywords_df[
            source_keywords_df["source"] == source_name
        ]
        .sort_values("mean_tfidf", ascending=False)
        .head(15)
        .sort_values("mean_tfidf")
    )

    # Create a horizontal bar chart.
    plt.figure(figsize=(10, 6))

    plt.barh(
        plot_data["keyword"],
        plot_data["mean_tfidf"]
    )

    plt.title(f"Top TF-IDF Terms: {source_name}")
    plt.xlabel("Mean TF-IDF score")
    plt.ylabel("Keyword or phrase")
    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# TOPIC MODELLING USING NMF
# ============================================================

# NMF stands for Non-negative Matrix Factorization.
#
# It decomposes the TF-IDF matrix into two smaller matrices:
#
# W: article-topic weights
#    Each row corresponds to an article.
#    Each column corresponds to a topic.
#
# H: topic-term weights
#    Each row corresponds to a topic.
#    Each column corresponds to a term.
#
# The algorithm discovers word patterns; humans interpret
# those patterns and give topics meaningful names.

# Check that there are enough documents and terms to fit NMF.
if X_tfidf.shape[0] < N_TOPICS:
    raise ValueError(
        "There are fewer articles than requested topics. "
        "Reduce N_TOPICS."
    )

if X_tfidf.shape[1] == 0:
    raise ValueError(
        "The TF-IDF vocabulary is empty. "
        "Check the text and vectorizer settings."
    )

# Fit the NMF model.
nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    max_iter=500
)

W = nmf_model.fit_transform(X_tfidf)
H = nmf_model.components_

print("NMF topic modelling complete.")
print("Article-topic matrix shape:", W.shape)
print("Topic-term matrix shape:", H.shape)


In [ ]:

# ============================================================
# EXTRACT TOP WORDS FOR EACH TOPIC
# ============================================================

topic_rows = []

for topic_index, topic_weights in enumerate(H):

    # Find the terms with the highest weights in this topic.
    sorted_indices = topic_weights.argsort()[::-1]

    top_indices = sorted_indices[:TOP_WORDS_PER_TOPIC]

    top_words = [
        terms[i] for i in top_indices
    ]

    top_weights = [
        round(float(topic_weights[i]), 4)
        for i in top_indices
    ]

    topic_rows.append({
        "topic_id": topic_index,
        "top_words": ", ".join(top_words),
        "top_word_weights": ", ".join(map(str, top_weights))
    })

topics_df = pd.DataFrame(topic_rows)

print("Discovered topics:")
display(topics_df)


In [ ]:

# ============================================================
# ASSIGN THE DOMINANT TOPIC TO EACH ARTICLE
# ============================================================

# W contains the weight of every topic for every article.
#
# The dominant topic is the topic with the largest weight.
# An article can still be associated with multiple topics.

df["dominant_topic"] = W.argmax(axis=1)

# Save the weight of the strongest topic for each article.
df["topic_strength"] = W.max(axis=1)

# Add readable top words for each article's dominant topic.
topic_word_lookup = dict(
    zip(topics_df["topic_id"], topics_df["top_words"])
)

df["dominant_topic_words"] = df["dominant_topic"].map(
    topic_word_lookup
)

print("Dominant topics assigned to articles.")

display(
    df[
        [
            "source",
            "title",
            "dominant_topic",
            "topic_strength",
            "dominant_topic_words"
        ]
    ].head(15)
)


In [ ]:

# ============================================================
# COMPARE TOPIC DISTRIBUTIONS BETWEEN SOURCES
# ============================================================

# Count how many articles from each source are assigned
# to each dominant topic.
topic_counts = pd.crosstab(
    df["source"],
    df["dominant_topic"]
)

# Convert counts into percentages within each source.
topic_percentages = (
    topic_counts
    .div(topic_counts.sum(axis=1), axis=0)
    * 100
)

print("Number of articles per topic and source:")
display(topic_counts)

print("Percentage of articles per topic and source:")
display(topic_percentages.round(2))

# Plot the percentage distribution.
topic_percentages.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Dominant Topic Distribution by Source")
plt.xlabel("Source")
plt.ylabel("Articles assigned to topic (%)")
plt.xticks(rotation=0)
plt.legend(title="Topic ID")
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# EXPORT RESULTS FOR MEMBER 4 AND THE GITHUB REPOSITORY
# ============================================================

# Create the results folder if it does not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Keywords for each article.
article_keywords_df.to_csv(
    OUTPUT_DIR / "article_keywords.csv",
    index=False
)

# 2. Average TF-IDF keyword scores for each source.
source_keywords_df.to_csv(
    OUTPUT_DIR / "source_keywords.csv",
    index=False
)

# 3. Top words associated with each topic.
topics_df.to_csv(
    OUTPUT_DIR / "topic_words.csv",
    index=False
)

# 4. Article dataset with dominant topic assignments.
df.to_csv(
    OUTPUT_DIR / "articles_with_topics.csv",
    index=False
)

# 5. Topic counts and percentages by source.
topic_counts.to_csv(
    OUTPUT_DIR / "topic_counts_by_source.csv"
)

topic_percentages.to_csv(
    OUTPUT_DIR / "topic_percentages_by_source.csv"
)

print("All result files have been saved in:", OUTPUT_DIR)
print("Files created:")

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", path.name)


In [ ]:

# ============================================================
# DOWNLOAD RESULTS FROM GOOGLE COLAB
# ============================================================

from google.colab import files

# Download whichever file you need.
# Run one line at a time and uncomment the desired file.

# files.download("results/article_keywords.csv")
# files.download("results/source_keywords.csv")
# files.download("results/topic_words.csv")
# files.download("results/articles_with_topics.csv")
# files.download("results/topic_counts_by_source.csv")
# files.download("results/topic_percentages_by_source.csv")
